# Demonstration Notebook: Spectrum 3D

This notebook demonstrates the generalized app API when looking at 3D Spectra/Spectral Cubes in the Notebook setting. This notebook follows the examples of the CubevizExample notebook and refers to the plugins used by Cubeviz. UI equivalents for these actions, as well as additional documentation about the Cubeviz configuration generally, can be found here: https://jdaviz.readthedocs.io/en/latest/cubeviz. These generalized workflows are supported as of Jdaviz v5.0.

### PLEASE NOTE
Running cells in rapid succession may cause errors or other bugs in the app. If these occur, you may need to rerun the cell or otherwise restart the notebook.

In [ ]:
import warnings
# Ignore warnings when loading data
warnings.simplefilter('ignore')

from photutils.aperture import CircularAperture
from regions import PixCoord, CirclePixelRegion

## Launch App

We create a Jdaviz app instance.

Multiple Jdaviz app instances can be created and used simultaneously if desired. See https://jdaviz.readthedocs.io/en/latest/quickstart.html#multiple-jdaviz-instances for more details.

In [ ]:
# Calling jd will implicitly create a jdaviz app instance
import jdaviz as jd

# Another option is to use gca() which refers to 'get current app'.
# It is not strictly necessary here but can be useful
# if multiple instances of the app are instantiated.
# jd = jdaviz.gca()

## Display App

This will show the app inline in the notebook. You can pass an integer to the `height` keyword to change the displayed height of the app (the default is 600). You can also pass `loc="popout:window"` to immediately pop out the Jdaviz app into a separate window.

In [ ]:
jd.show()

## Load Data into the App

Now we load observations. If you already have the files on your local machine, you can 
load them by passing their file paths to `load` as strings. For this example, 
we will retrieve remote data from [MAST](https://mast.stsci.edu/) via `astroquery`
by passing the observation's URI as a string. By default, the downloaded files are 
saved to the current working directory. If you set `cache=False` in the `load` 
call, it will re-download the file if desired.

Note, when retrieving MAST data through astroquery, it is cached by default. 
It is possible for files to be updated in MAST with more recent calibrations
but remain the same size, in which case your local cached file would not be replaced by the new
version. Again, you can turn off caching by setting `cache=False` to force it to re-download the 
file.

In [ ]:
data_label = "jw02732-c1001_t004_miri_ch1-short_s3d"
uri = f"mast:JWST/product/{data_label}.fits"
jd.load(uri, format='3D Spectrum', cache=True)

## Access Viewers

For the default data, the three viewers represent the flux cube, the uncertainty cube, and the 1D spectral extraction from the flux cube. 

In the cubeviz configuration, these viewers are labeled under the assumption of this data structure. Given that the app is generalized, default viewer labels are are agnostic to these assumptions.

In [ ]:
flux_viewer = jd.viewers['3D Spectrum']
unc_viewer = jd.viewers['3D Spectrum (1)']
sp_ext_viewer = jd.viewers['1D Spectrum']

Note also that in the above example there are mouse-over coordinates visible by default.

## Access Data

We can access information about the data by calling `jd.datasets` which returns a dictionary with the labels as keys, and `DataApi` objects as the values. We can retrieve data from the `DataApi` objects by calling `data_api_object.get_data()`.

In [ ]:
jd.datasets

## Work with Subsets

It possible to make selections/regions in images and export these to Astropy regions. 

### User Task

Click on the circular selection tool (<img
  src="data:image/svg+xml;utf8,<svg xmlns='http://www.w3.org/2000/svg' viewBox='0 0 23.53 24.66' aria-hidden='true'><path d='M14.44,7.27a7,7,0,1,0,7,7A7,7,0,0,0,14.44,7.27Zm-1,11h0A1.48,1.48,0,0,1,12,16.76h0a1.48,1.48,0,0,1,1.48-1.48h0a1.48,1.48,0,0,1,1.48,1.48h0A1.48,1.48,0,0,1,13.44,18.24Zm2.11-4.5h0a1.48,1.48,0,0,1-1.48-1.48h0a1.48,1.48,0,0,1,1.48-1.48h0A1.48,1.48,0,0,1,17,12.26h0A1.48,1.48,0,0,1,15.55,13.74Z' fill='%23000' stroke='%23fff' stroke-width='1.5' stroke-linejoin='round' stroke-linecap='round' vector-effect='non-scaling-stroke' style='paint-order:stroke fill'/><rect x='2.48' y='16.18' width='2.96' height='2.96' rx='1.48' fill='%23000' stroke='%23fff' stroke-width='1.5' stroke-linejoin='round' vector-effect='non-scaling-stroke' style='paint-order:stroke fill'/><rect x='4.48' y='3.3' width='2.96' height='2.96' rx='1.48' fill='%23000' stroke='%23fff' stroke-width='1.5' stroke-linejoin='round' vector-effect='non-scaling-stroke' style='paint-order:stroke fill'/><rect x='9.47' y='1.78' width='2.96' height='2.96' rx='1.48' fill='%23000' stroke='%23fff' stroke-width='2' stroke-linejoin='round' vector-effect='non-scaling-stroke' style='paint-order:stroke fill'/></svg>"
  width="16"
  style="vertical-align:-4px;"
/>), and drag and click to select an interesting region on the sky. We can then export this region with:

In [ ]:
regions = jd.plugins['Subset Tools'].get_regions(region_type="spatial")

In [ ]:
regions

It is also possible to programmatically pass a `regions` shape or a `photutils` aperture shape.

*NOTE: Sky regions are not yet supported for data cubes.*

In [ ]:
# photutils aperture
my_aper = CircularAperture((15, 10), r=5)

# regions shape
my_reg = CirclePixelRegion(center=PixCoord(x=35, y=35), radius=10)

my_regions = [my_aper, my_reg]
jd.plugins['Subset Tools'].import_region(my_regions, combination_mode='new')

To get the spectra, we use `jd.datasets` again given that the regions we created are mapped to 1D spectra objects created by collapsing the cube in wavelength space over the selected region. These can be seen in the 1D Spectrum Viewer, you may have to zoom out or otherwise click the <!-- ../jdaviz/data/icons/home.svg -->
<img src="data:image/svg+xml;utf8,<svg%20xmlns%3D%22http://www.w3.org/2000/svg%22%20id%3D%22Layer_1%22%20data-name%3D%22Layer%201%22%20viewBox%3D%220%200%2023.53%2024.66%22><defs%20/><path%20d%3D%22M9.41%2C21.77V14.71h4.71v7.06H20V12.36h3.53L11.77%2C1.77%2C0%2C12.36H3.53v9.41Z%22%20fill%3D%22%23000000%22%20stroke%3D%22%23ffffff%22%20stroke-width%3D%221.75%22%20stroke-linecap%3D%22round%22%20stroke-linejoin%3D%22round%22%20vector-effect%3D%22non-scaling-stroke%22%20style%3D%22paint-order:%20stroke%20fill%22%20/></svg>" width="14" style="vertical-align:-2px;" /> icon.

These *cannot* be retreived from the `Subset Tools` plugin since they are represented in the app as full spectra.

In [ ]:
# Returns the subsets as specutils.Spectrum objects
subset_1_spectrum = jd.datasets['Spectrum (Subset 1, sum)'].get_data()
subset_1_spectrum

In [ ]:
subset_2_spectrum = jd.datasets['Spectrum (Subset 2, sum)'].get_data()
subset_2_spectrum